In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [25]:
spark = SparkSession.builder.appName('09').getOrCreate()

## **목표**: 함께 등장한 hero가 가장 많은 주인공 hero 찾기

- `Marvel-names.txt`
> - `id`:**int**, `name`:**str** 구조, `sep`는 공백
> - 일반적인 행과 열이 있는 table형태이므로 `read.csv`로 로드함

In [26]:
# schema 지정
names_schema = StructType([
    StructField('id',IntegerType(),True),
    StructField('name',StringType(),True)
])

In [27]:
names = spark.read.csv('C:/Users/User/Downloads/Marvel-names.txt',
                       schema=names_schema,
                       sep=' ')
names.show(5)
names.printSchema()

+---+--------------------+
| id|                name|
+---+--------------------+
|  1|24-HOUR MAN/EMMANUEL|
|  2|3-D MAN/CHARLES CHAN|
|  3|    4-D MAN/MERCURIO|
|  4|             8-BALL/|
|  5|                   A|
+---+--------------------+
only showing top 5 rows

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)



- `Marvel-graph.txt`
> - 각 행은 한편의 comics를 의미함
> - 각 행의 첫번째 숫자는 해당 comics의 주인공 hero의 ID
> - 두번째 숫자부터는 함께 등장한 hero의 ID
> - 열을 구분하지 않고, 로드할 때는 `read.text`를 사용하며, 결과는 `value`열 하나를 갖는 `DataFrame을 반환함`

In [28]:
graph = spark.read.text('C:/Users/User/Downloads/Marvel-graph.txt')
graph.show(5)
graph.printSchema()

+--------------------+
|               value|
+--------------------+
|5988 748 1722 375...|
|5989 4080 4264 44...|
|5982 217 595 1194...|
|5983 1165 3836 43...|
|5980 2731 3712 15...|
+--------------------+
only showing top 5 rows

root
 |-- value: string (nullable = true)



- **step 1**: 각 Row의 첫번째 값인 주인공 hero id를 분리하여 새로운 열에 할당함
> - `split(col,sep)`: `col`열의 값(`str`)을 `sep`을 기준으로 분리하여, `list`로 반환
>> - 이 과정에서 `value`열의 첫번째 값을 추출해야 하므로 `split(value,' ')[0]`을 적용
> - 문자열을 분리하여 값을 추출할 때의 주의 사항
>> - 문자열의 좌우에 보이지않는 공백이 포함되어 있을 수 있으므로 `trim(col)`을 통해 공백을 삭제하는 과정을 반드시 거치는 것이 좋음

In [29]:
graph = graph.withColumn('id',split(trim(graph.value),' ')[0])
graph.show(5)
graph.printSchema()

+--------------------+----+
|               value|  id|
+--------------------+----+
|5988 748 1722 375...|5988|
|5989 4080 4264 44...|5989|
|5982 217 595 1194...|5982|
|5983 1165 3836 43...|5983|
|5980 2731 3712 15...|5980|
+--------------------+----+
only showing top 5 rows

root
 |-- value: string (nullable = true)
 |-- id: string (nullable = true)



- **step 2**: 각 행에 포함된 hero의 수를 count하여 새로운 열에 할당함
> - `split(value,' ')`을 통해 값을 분리한 후, 반환된 `list`에 포함된 값의 수를 찾음
> - `size(col)`: `col`의 값이 `list`인 경우, 해당 `list`가 포함하고 있는 **원소의 수**를 반환

In [30]:
graph_count = graph.withColumn('count',size(split(trim(graph.value),' '))-1)
# 1. trim(graph.value): graph.value 값(문자열)의 좌우 공백을 삭제
# 2. split(trim(graph.value),' '): 
#    1의 결과로부터 문자열을 공백(' ')을 기준으로 분리하여 list로 반환
# 3. size(split(trim(graph.value),' ')):
#    2의 결과로부터 반환된 list에 속한 원소의 수를 반환
# 4. size(split(trim(graph.value),' '))-1:
#    -1을 하는 이유는 첫번째 주인공 hero는 count에서 제외하기 위함임
graph_count.show(5)

+--------------------+----+-----+
|               value|  id|count|
+--------------------+----+-----+
|5988 748 1722 375...|5988|   48|
|5989 4080 4264 44...|5989|   40|
|5982 217 595 1194...|5982|   42|
|5983 1165 3836 43...|5983|   14|
|5980 2731 3712 15...|5980|   24|
+--------------------+----+-----+
only showing top 5 rows



- **step 3**: 사용하지 않는 `value`열 삭제

In [31]:
graph_count = graph_count.drop('value')
graph_count.show(5)

+----+-----+
|  id|count|
+----+-----+
|5988|   48|
|5989|   40|
|5982|   42|
|5983|   14|
|5980|   24|
+----+-----+
only showing top 5 rows



- **step 4**: 각 주인공 hero 별 함께 등장한 hero의 수 합을 찾음
> - 각 행은 comics 한 편을 의미하며, hero는 여러편의 comics 주인공으로 등장할 수 있으므로, id 별 집계가 필요함
> - 주인공 hero 별 `groupBy` 적용, `count`의 합산

In [33]:
hero_connection = graph_count.groupBy('id').agg(sum('count').alias('connection'))
hero_connection.show(5)

+----+----------+
|  id|connection|
+----+----------+
| 691|         6|
|1159|        11|
|3959|       142|
|1572|        35|
|2294|        14|
+----+----------+
only showing top 5 rows



- **step 5**: `connection`이 가장 큰 hero를 찾아야 하므로 `connection`을 기준으로 정렬함
> - `connection`이 가장 큰 hero ID를 `pupular_hero_id`에 저장함

In [34]:
hero_connection.orderBy(hero_connection.connection.desc()).show(5)

+----+----------+
|  id|connection|
+----+----------+
| 859|      1933|
|5306|      1741|
|2664|      1528|
|5716|      1426|
|6306|      1394|
+----+----------+
only showing top 5 rows



In [36]:
popular_hero_id = hero_connection.orderBy(hero_connection.connection.desc()).\
collect()[0]['id']
print(popular_hero_id)

859


- **step 6**: `names` dataframe으로부터 `popular_hero_id` hero의 이름을 찾음

In [37]:
names.filter(names.id == popular_hero_id).show()

+---+---------------+
| id|           name|
+---+---------------+
|859|CAPTAIN AMERICA|
+---+---------------+



In [38]:
popular_hero = names.filter(names.id == popular_hero_id).collect()[0]['name']
print(popular_hero)

CAPTAIN AMERICA
